[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/prefect-certified/notebooks/day-02-flows-tasks.ipynb#scrollTo=aa11bb22)

---
# Day 2 · Flows and Tasks — The Building Blocks
**certified-journeys / prefect-certified** · Day 2 · Learn

> **Goal for today:** Decompose a flow into `@task`-decorated units, understand the flow-vs-task distinction, pass return values between tasks, and use run names to trace exactly which code ran in the Prefect UI.


In [ ]:
%pip install -q prefect


## Step 1 · The `@task` decorator — turning a function into an observable unit

In Prefect, a **flow** is the top-level orchestrator and a **task** is an individual unit of work inside it.

| | `@flow` | `@task` |
|---|---|---|
| Role | Orchestrate the overall pipeline | Execute a discrete piece of work |
| Can call other flows? | Yes (subflows) | No |
| Can call tasks? | Yes | Yes (within a flow context) |
| Retry granularity | Retries the entire flow body | Retries just that task |
| State tracked separately? | Yes — flow run | Yes — task run (nested inside) |
| Cacheable? | No | Yes — `cache_key_fn` |

The key insight: **flows orchestrate, tasks execute**. Keep tasks small and single-purpose so you get fine-grained retry, caching, and observability.


In [ ]:
from prefect import flow, task, get_run_logger

# A minimal task — one decorator, no other changes
@task(name="fetch-posts-task")
def fetch_posts(limit: int = 5) -> list:
    """Fetch sample posts from JSONPlaceholder (free mock REST API)."""
    import json, urllib.request
    url = f"https://jsonplaceholder.typicode.com/posts?_limit={limit}"
    with urllib.request.urlopen(url) as resp:
        return json.loads(resp.read())

@flow(name="first-task-flow", log_prints=True)
def first_task_flow():
    """Simplest possible flow that contains a task."""
    logger = get_run_logger()
    posts = fetch_posts(limit=3)          # calling a task from a flow — creates a task run
    logger.info(f"Got {len(posts)} posts")
    return posts

result = first_task_flow()
print(f"Returned {len(result)} posts")
print(f"First post title: {result[0]['title'][:50]}")


### What just happened?

- **`@task`** created a separate tracked run inside the flow run — you now have both a *flow run* and a *task run* in the Prefect state store.
- The `name="fetch-posts-task"` argument sets the display name in the UI; without it, Prefect uses the function name.
- **Calling a task from a flow** (`fetch_posts(limit=3)`) is identical to calling a normal function — Prefect wraps the call transparently.
- In the Prefect UI (Runs view), you can expand the flow run to see each task run nested inside it, including its individual state, duration, and logs.


## Step 2 · Building a multi-task flow — orchestrate, don't execute

A well-structured Prefect flow follows a clear division of labour:
- The **flow function** is thin — it just calls tasks in order and passes results between them.
- The **task functions** contain all business logic.

This separation means you can:
1. Retry individual tasks without re-running the whole flow.
2. Cache expensive tasks so they skip on re-runs.
3. See exactly which step failed in the UI without reading logs.

Think of the flow as a **recipe** and tasks as **cooking steps** — the recipe lists steps in order, each step does one thing.


In [ ]:
import json
import urllib.request
from datetime import datetime
from prefect import flow, task, get_run_logger

# ─── Tasks: each does exactly ONE thing ───────────────────────────────────────

@task(name="extract-posts")
def extract_posts(user_id: int) -> list:
    """Extract: fetch raw posts from the API for one user."""
    url = f"https://jsonplaceholder.typicode.com/posts?userId={user_id}"
    with urllib.request.urlopen(url) as resp:
        return json.loads(resp.read())

@task(name="transform-posts")
def transform_posts(posts: list) -> list:
    """Transform: clean and reshape the raw data."""
    return [
        {
            "id":    p["id"],
            "title": p["title"].title(),            # normalise capitalisation
            "words": len(p["body"].split()),         # word count for the post body
        }
        for p in posts
    ]

@task(name="load-summary")
def load_summary(transformed: list, user_id: int) -> dict:
    """Load: produce a final summary (in production this might write to a DB)."""
    return {
        "user_id":    user_id,
        "post_count": len(transformed),
        "avg_words":  round(sum(p["words"] for p in transformed) / len(transformed), 1),
        "loaded_at":  datetime.utcnow().isoformat(),
    }

# ─── Flow: thin orchestrator — calls tasks, passes results between them ───────

@flow(name="etl-pipeline", log_prints=True)
def etl_pipeline(user_id: int = 1):
    """Classic ETL: extract → transform → load, each step a separate task."""
    raw       = extract_posts(user_id)          # return value is a plain Python list
    cleaned   = transform_posts(raw)             # pass the list directly — creates a dependency
    summary   = load_summary(cleaned, user_id)   # dependency: load runs after transform
    print(f"Summary: {summary}")
    return summary

summary = etl_pipeline(user_id=2)
print(f"\nUser 2 — {summary['post_count']} posts, avg {summary['avg_words']} words/post")


### What just happened?

- Three task runs were created — **extract → transform → load** — each tracked separately inside the flow run.
- **Passing a return value from one task as input to the next** is how Prefect knows about data dependencies. No explicit DAG definition required.
- If `transform_posts` had failed, Prefect would have marked its task run `Failed` and propagated the failure to the flow run — `load_summary` would never have been called.
- In the Prefect UI **Runs > [this run] > Task Runs** you can see all three task runs listed with their individual states and durations.


## Step 3 · Task configuration — names, retries, and tags

The `@task` decorator accepts configuration that controls retry behaviour, UI display, and caching:

| Parameter | Type | What it does |
|---|---|---|
| `name` | `str` | Display name in the UI task run list |
| `description` | `str` | Shown in UI sidebar |
| `retries` | `int` | Retry this task N times before failing |
| `retry_delay_seconds` | `int \| float` | Wait between retries |
| `tags` | `list[str]` | Filter / group task runs in UI by tag |
| `timeout_seconds` | `int` | Kill this task after N seconds |
| `log_prints` | `bool` | Capture `print()` in Prefect logs |

Task-level retries are more granular than flow-level retries — only the failed step reruns, not the entire pipeline.


In [ ]:
import json
import urllib.request
from prefect import flow, task

# Simulate a flaky network task that needs retries
_call_count = {"n": 0}  # module-level counter to track retry attempts

@task(
    name="flaky-fetch",
    description="Fetches posts; simulates transient failures to demo retries",
    retries=3,                    # retry up to 3 times
    retry_delay_seconds=0.5,      # short delay for demo; use 30–120 s in production
    tags=["network", "extract"],  # tags visible in the UI for filtering
    log_prints=True,
)
def flaky_fetch(fail_until_attempt: int = 2) -> list:
    """Fetch posts, but pretend the first N-1 attempts fail."""
    _call_count["n"] += 1
    attempt = _call_count["n"]
    print(f"  attempt #{attempt}")

    if attempt < fail_until_attempt:
        raise ConnectionError(f"Simulated network error on attempt {attempt}")

    # Succeed on the required attempt
    url = "https://jsonplaceholder.typicode.com/posts?_limit=3"
    with urllib.request.urlopen(url) as resp:
        return json.loads(resp.read())

@flow(name="retry-demo-flow")
def retry_demo():
    """Flow that calls a flaky task — Prefect retries automatically."""
    posts = flaky_fetch(fail_until_attempt=2)  # fails once, succeeds on attempt 2
    return len(posts)

count = retry_demo()
print(f"\nFlow completed — fetched {count} posts after {_call_count['n']} attempt(s)")


### What just happened?

- The task failed on attempt 1, was **automatically retried** after 0.5 s, and succeeded on attempt 2.
- **Only the failing task retried** — the flow run itself stayed in `Running` state throughout.
- **`tags=["network", "extract"]`** lets you filter task runs in the Prefect UI by tag, useful when a flow has many tasks.
- In production, set `retry_delay_seconds` to 30–120 s to give upstream services (APIs, databases) time to recover before retrying.


## Step 4 · Run names — tracing exactly which code ran in the UI

By default, Prefect generates random poetic names for flow runs (`capable-turtle`, `radiant-fox`, …). You can override this with `flow_run_name` and `task_run_name` to embed runtime values — user IDs, dates, file names — directly into the run name.

This makes it trivial to find a specific execution in the UI without reading logs:

```
etl-for-user-7-on-2024-01-15     ← flow run name with injected values
  ├── extract-user-7              ← task run name
  ├── transform-7-posts
  └── load-user-7-summary
```

Both `flow_run_name` and `task_run_name` accept Python f-string-style templates using the function's parameter names.


In [ ]:
import json
import urllib.request
from datetime import date
from prefect import flow, task, get_run_logger

# task_run_name uses the task's parameter names as template variables
@task(name="extract", task_run_name="extract-user-{user_id}")
def extract(user_id: int) -> list:
    url = f"https://jsonplaceholder.typicode.com/posts?userId={user_id}"
    with urllib.request.urlopen(url) as resp:
        return json.loads(resp.read())

@task(name="summarise", task_run_name="summarise-{n_posts}-posts-for-user-{user_id}")
def summarise(posts: list, user_id: int) -> dict:
    n_posts = len(posts)
    return {"user_id": user_id, "count": n_posts}

# flow_run_name also accepts a callable (called once per run)
@flow(
    name="named-etl",
    flow_run_name="etl-user-{user_id}-{today}",  # injected from flow parameters
    log_prints=True,
)
def named_etl(user_id: int = 1, today: str = str(date.today())):
    """Flow and task runs with descriptive names for easy UI tracing."""
    logger = get_run_logger()

    # Access the current flow run's name at runtime
    from prefect.context import get_run_context
    ctx = get_run_context()
    logger.info(f"Flow run name: {ctx.flow_run.name}")

    posts   = extract(user_id)
    summary = summarise(posts, user_id)
    print(f"Done: {summary}")
    return summary

result = named_etl(user_id=3)
print(f"Result: {result}")


### What just happened?

- **`flow_run_name`** embedded `user_id` and today's date — making this run instantly identifiable in the UI without opening its logs.
- **`task_run_name`** used the task's parameter names as template variables — note that `summarise` uses `n_posts` which is computed *inside* the function body, not a parameter; for that use a callable instead of a template string.
- `get_run_context().flow_run.name` lets you read the run's name at runtime — useful for logging or writing to a results table.
- In production: use names like `etl-{env}-{source}-{date}` so on-call engineers can filter the Runs view by environment or source without reading every log.


## Step 5 · Passing data between tasks — creating explicit dependencies

The most important pattern in Prefect: **return a value from one task and pass it as an argument to the next**. This creates an explicit data dependency — Prefect (and you) both know the second task cannot run until the first completes.

```
raw_data  = extract()          # Task A completes, returns value
cleaned   = transform(raw_data) # Task B receives A's output — dependency created
result    = load(cleaned)       # Task C receives B's output — chained dependency
```

This differs from implicit dependencies (`wait_for`) covered in Day 3 — data-passing dependencies carry a value. Implicit dependencies just say "run B after A" with no data transfer.


In [ ]:
import json
import urllib.request
from prefect import flow, task, get_run_logger

# Four tasks chained: each receives the previous task's output

@task(name="fetch-users")
def fetch_users(limit: int = 3) -> list:
    """Fetch a list of user objects."""
    url = f"https://jsonplaceholder.typicode.com/users?_limit={limit}"
    with urllib.request.urlopen(url) as resp:
        return json.loads(resp.read())

@task(name="extract-emails")
def extract_emails(users: list) -> list:
    """Extract email addresses from user objects (receives output of fetch_users)."""
    return [u["email"].lower() for u in users]

@task(name="validate-emails")
def validate_emails(emails: list) -> dict:
    """Validate format — mark each email valid/invalid."""
    results = {}
    for email in emails:
        # Simple structural check: must have exactly one '@' and a '.' after it
        parts = email.split("@")
        results[email] = len(parts) == 2 and "." in parts[1]
    return results

@task(name="build-report")
def build_report(validation: dict) -> dict:
    """Build a summary report from the validation results."""
    valid   = [e for e, ok in validation.items() if ok]
    invalid = [e for e, ok in validation.items() if not ok]
    return {"valid": valid, "invalid": invalid, "pass_rate": len(valid) / len(validation)}

@flow(name="email-validation-pipeline", log_prints=True)
def email_validation_pipeline(n_users: int = 3):
    """Four-task chain: fetch → extract → validate → report."""
    logger = get_run_logger()

    users      = fetch_users(n_users)          # returns list of user dicts
    emails     = extract_emails(users)          # receives list → returns list of strings
    validation = validate_emails(emails)        # receives list → returns dict
    report     = build_report(validation)       # receives dict → returns summary

    logger.info(f"Pass rate: {report['pass_rate']:.0%}")
    print(f"Valid emails:   {report['valid']}")
    print(f"Invalid emails: {report['invalid']}")
    return report

report = email_validation_pipeline(n_users=5)
print(f"\nFinal pass rate: {report['pass_rate']:.0%}")


### What just happened?

- Four task runs were created in sequence — each **received the previous task's return value as a Python object**.
- Prefect knew to run them in order because the inputs of each task depend on the outputs of the previous one.
- In the **Prefect UI task run graph** (Runs → this run → Graph view), you can see the directed edges between task nodes representing these data dependencies.
- **If any task failed**, all downstream tasks that depend on its output would be automatically skipped — their state would be `NotReady`.


## Step 6 · Inspecting the task run graph in the UI

Every flow run with tasks has a **Graph view** in the Prefect UI that visualises the execution DAG.

**How to find it:**
1. Start the server: `prefect server start`
2. Open **http://127.0.0.1:4200**
3. Click **Runs** in the left sidebar
4. Click the flow run you want to inspect
5. Select the **Graph** tab (next to Timeline and Logs)

**What you see:**
- Nodes = task runs, coloured by state (`Completed` = green, `Failed` = red)
- Edges = data dependencies (created by passing return values between tasks)
- Hovering a node shows its duration, state, and a direct link to its logs

The code below runs a flow whose structure produces a clear graph — run it, then view it in the UI.


In [ ]:
import json
import urllib.request
from prefect import flow, task
from prefect.context import get_run_context

@task(name="fetch-todos")
def fetch_todos(limit: int = 10) -> list:
    url = f"https://jsonplaceholder.typicode.com/todos?_limit={limit}"
    with urllib.request.urlopen(url) as resp:
        return json.loads(resp.read())

@task(name="split-by-status")
def split_by_status(todos: list) -> tuple:
    """Split the todo list into completed and pending sub-lists."""
    done    = [t for t in todos if t["completed"]]
    pending = [t for t in todos if not t["completed"]]
    return done, pending

@task(name="count-completed")
def count_completed(done: list) -> int:
    return len(done)

@task(name="count-pending")
def count_pending(pending: list) -> int:
    return len(pending)

@task(name="build-stats")
def build_stats(n_done: int, n_pending: int) -> dict:
    total = n_done + n_pending
    return {"done": n_done, "pending": n_pending, "pct_done": round(n_done / total * 100, 1)}

@flow(name="todo-stats-pipeline", flow_run_name="todo-stats-run", log_prints=True)
def todo_stats_pipeline(limit: int = 10):
    """Pipeline with a fan-out shape: split → (count_done, count_pending) → stats."""
    ctx = get_run_context()
    print(f"Flow run name: {ctx.flow_run.name}")

    todos           = fetch_todos(limit)
    done, pending   = split_by_status(todos)    # fan-out: two branches depend on this
    n_done          = count_completed(done)
    n_pending       = count_pending(pending)    # runs in parallel with count_completed
    stats           = build_stats(n_done, n_pending)  # fan-in: waits for both counts

    print(f"Stats: {stats}")
    return stats

stats = todo_stats_pipeline(limit=20)
print(f"\nDone: {stats['done']}, Pending: {stats['pending']}, "
      f"Completion rate: {stats['pct_done']}%")
print("\nView the task run graph: prefect server start → http://127.0.0.1:4200 → Runs → Graph")


### What just happened?

- The flow has a **fan-out / fan-in shape**: `split_by_status` produces two outputs that feed two parallel tasks, which then converge at `build_stats`.
- In the Graph view this appears as a diamond shape — `split_by_status` → `count_completed` + `count_pending` → `build_stats`.
- **`flow_run_name="todo-stats-run"`** gives the run a stable name — easy to find in the Runs list after multiple executions.
- The `ctx.flow_run.name` pattern lets you embed the run name into any downstream artifact (e.g. a database row or file name).


In [ ]:
# Challenge: Build a 3-task flow from scratch
#
# Requirements:
#   1. Task 1 — `fetch_comments(post_id)`: fetch comments for a post from
#      https://jsonplaceholder.typicode.com/comments?postId={post_id}
#   2. Task 2 — `extract_names(comments)`: return a list of commenter names
#      (hint: each comment has a "name" field)
#   3. Task 3 — `build_summary(names, post_id)`: return a dict with
#      keys: post_id, commenter_count, names
#   4. A flow `comment_pipeline(post_id=1)` that chains all three tasks
#      and uses flow_run_name="comments-post-{post_id}"
#   5. Print the summary at the end of the flow
#
# Scaffold:

import json
import urllib.request
from prefect import flow, task

# TODO: implement Task 1
# @task(name="fetch-comments")
# def fetch_comments(post_id: int) -> list:
#     ...

# TODO: implement Task 2
# @task(name="extract-names")
# def extract_names(comments: list) -> list:
#     ...

# TODO: implement Task 3
# @task(name="build-summary")
# def build_summary(names: list, post_id: int) -> dict:
#     ...

# TODO: implement the flow
# @flow(name="comment-pipeline", flow_run_name="comments-post-{post_id}", log_prints=True)
# def comment_pipeline(post_id: int = 1):
#     ...

# Uncomment once implemented:
# result = comment_pipeline(post_id=2)
# print(result)


---
## Day 2 key concepts recap

| Concept | What to remember |
|---|---|
| `@task` decorator | Marks an individual unit of work; creates a separate tracked task run |
| Flows orchestrate, tasks execute | Keep the flow function thin; put all logic in tasks |
| Data dependency | Pass a task's return value as an arg to the next task — Prefect infers order |
| `task_run_name` | Template string using task parameter names; makes runs traceable in the UI |
| `flow_run_name` | Same pattern for the flow run itself; embed env, user IDs, dates |
| Task retries | `retries=N` + `retry_delay_seconds=N` — only the failing task retries |
| `tags` | Filter/group task runs in the UI; useful for large flows |
| Graph view | Runs → [flow run] → Graph — shows the dependency DAG with state colours |

> **Tip:** Keep tasks small and single-purpose — a task that does one thing is easy to retry, cache, and test independently.

---
## What's next
**Day 3** → Use `task.submit()` and `task.map()` for concurrent execution, collect results with `PrefectFuture.result()`, and express non-data dependencies with `wait_for`.

Mark Day 2 complete in your [tracker](../index.html).
